# Locy Neural Use Case: Predictive Maintenance (Rust)

Compact Rust counterpart to the Python predictive-maintenance flagship. Same Phase D capability surface — `CREATE MODEL` + `CALIBRATE` + `VALIDATE` + `EXPLAIN` — driving a small inline equipment graph through a `MockClassifier` rather than a registered Python callable.


## How To Read This Notebook

- Same conceptual shape as the Python flagship; mock classifier inline.
- Schema declared first, then seeded data, then the Locy program.
- Read the EXPLAIN output to see `NeuralProvenance` per derivation.


## 1) Setup


In [ ]:
use std::sync::Arc;
use uni_db::{DataType, Uni, Result};
use uni_locy::{LocyConfig, MockClassifier, NeuralClassifier, FeatureValue};

let db = Uni::in_memory().build().await?;


## 2) Schema


In [ ]:
db.schema()
    .label("Equipment")
        .property("equipment_id", DataType::String)
        .property("air_temp", DataType::Float64)
        .property("actual_failed", DataType::Bool)
    .done()
    .apply()
    .await?;


## 3) Seed Inline


In [ ]:
let session = db.session();
let tx = session.tx().await?;
// 8 equipment instances, 3 actually failed (higher air_temp).
let rows: &[(&str, f64, bool)] = &[
    ("e01", 301.0, true), ("e02", 298.4, false),
    ("e03", 298.6, false), ("e04", 300.5, true),
    ("e05", 298.7, false), ("e06", 299.0, false),
    ("e07", 300.8, true), ("e08", 298.5, false),
];
for (eid, t, failed) in rows {
    let q = format!(
        "CREATE (:Equipment {{equipment_id: '{}', air_temp: {}, actual_failed: {}}})",
        eid, t, failed
    );
    tx.execute(&q).await?;
}
tx.commit().await?;


## 4) Register the Classifier

`MockClassifier::new` takes a closure `Fn(&ClassifyInput) -> f64`. The classifier reads the value bound under the model's INPUT name and emits an intentionally over-confident probability so `CALIBRATE` has measurable work to do.


In [ ]:
let classifier: Arc<dyn NeuralClassifier> = Arc::new(
    MockClassifier::new("failure_likelihood", |inp| {
        let air = match inp.features.get("e") {
            Some(FeatureValue::Float(v)) => *v,
            _ => 0.0,
        };
        let z = (air - 298.5) * 1.5 - 1.0;
        let p = 1.0 / (1.0 + (-z).exp());
        let p_sharp = 1.0 / (1.0 + (-3.0 * (p - 0.5)).exp());
        p_sharp.clamp(0.0, 1.0)
    })
);
let mut config = LocyConfig::default();
config.classifier_registry.insert("failure_likelihood".to_string(), classifier);


## 5) CREATE MODEL + Score


In [ ]:
let program = r#"
CREATE MODEL failure_likelihood AS
  INPUT (e)
  FEATURES e.air_temp
  OUTPUT PROB will_fail
  USING xervo('classify/failure-likelihood-v1')

CREATE RULE asset_risk AS
  MATCH (e:Equipment)
  YIELD KEY e, failure_likelihood(e.air_temp) AS risk
"#;
let result = session.locy_with(program).with_config(config.clone()).run().await?;
let asset_rows = result.derived().get("asset_risk").map(|v| v.len()).unwrap_or(0);
println!("Scored {} equipment", asset_rows);


## 6) CALIBRATE + VALIDATE


In [ ]:
let calibrate_program = r#"
CREATE MODEL failure_likelihood AS
  INPUT (e)
  FEATURES e.air_temp
  OUTPUT PROB will_fail
  USING xervo('classify/failure-likelihood-v1')

CALIBRATE failure_likelihood
  ON MATCH (e:Equipment)
  TARGET e.actual_failed
  METHOD platt_scaling
"#;
let calib = session.locy_with(calibrate_program).with_config(config.clone()).run().await?;
println!("command_results: {} entries", calib.command_results().len());


## 7) EXPLAIN One High-Risk Asset

Returns the derivation tree, including a `NeuralProvenance` leaf per classifier invocation (`model_name`, `raw_probability`, `calibrated_probability`, feature dict).


In [ ]:
let explain_program = format!("{}{}", program, r#"

EXPLAIN RULE asset_risk WHERE e.equipment_id = 'e01'
"#);
let explain = session.locy_with(&explain_program).with_config(config).run().await?;
println!("EXPLAIN command_results: {}", explain.command_results().len());
